# Integración ejecutable de simuladores B2B/B2G

Esta versión vigente importa `cuidado.services.simulador_b2b_b2g`, la única implementación canónica. Conserva la procedencia de la entrega inicial de Azael y no genera ni mantiene otra copia de las fórmulas.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().resolve()
while root != root.parent and not (root / "cuidado").is_dir():
    root = root.parent
if not (root / "cuidado").is_dir():
    raise RuntimeError("No se encontró la raíz del proyecto")
sys.path.insert(0, str(root))

from cuidado.services.simulador_b2b_b2g import cargar_escenarios, ejecutar_escenarios

csv_path = root / "data" / "processed" / "simulador_b2b_b2g_parametros.csv"
print(root)
print(csv_path)

## Fórmulas y límites

- Ahorro bruto B2B: bajas evitadas multiplicadas por costos unitarios de rotación.
- Beneficio neto: no se calcula si falta `costo_intervencion`; no se convierte en cero.
- B2G separa ISN (impuesto) de IMSS e INFONAVIT (aportaciones patronales).
- Todas las entradas son sintéticas y deben validar fuentes, unidades, periodos y rangos.
- Los 30 escenarios son alternativas; no se suman como empresas distintas.

In [ ]:
escenarios = cargar_escenarios(csv_path)
assert len(escenarios) == 30
assert escenarios["escenario_id"].is_unique
print(f"Escenarios válidos: {len(escenarios)}")
escenarios[["escenario_id", "numero_empleadas", "salario_promedio", "bajas_anuales", "costo_intervencion"]].head()

In [ ]:
resultados = ejecutar_escenarios(csv_path)
assert len(resultados) == 30
assert resultados["b2b_beneficio_neto"].isna().all()
assert resultados[["b2g_imss_patronal", "b2g_isn_impuesto", "b2g_infonavit_aportacion"]].notna().all().all()
print("Beneficio neto: no calculable en los 30 escenarios")
print("Ahorro bruto, impuestos y aportaciones separados y validados")
resultados[["escenario_id", "b2b_ahorro_bruto", "b2b_beneficio_neto", "b2g_isn_impuesto", "b2g_imss_patronal", "b2g_infonavit_aportacion"]].head()